In [2]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor

from lineartree import (
    LinearTreeRegressor,
    SobolevForestRegressor,
)
import numpy as np
from sklearn.preprocessing import SplineTransformer
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

In [3]:
class TensorSplineRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, n_knots=6, degree=3, knots="quantile", r_alpha=None):
        self.n_knots = n_knots
        self.degree = degree
        self.knots = knots
        self.r_alpha = r_alpha

    def _basis(self, X, alpha):
        blocks = [
            spline(X[:, j], nu=alpha[j])
            for j, spline in enumerate(self.spline_.bsplines_)
        ]

        basis = blocks[0]
        for block in blocks[1:]:
            basis = np.einsum(
                "ni,nj->nij", basis, block
            ).reshape(len(X), -1)

        return basis

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.n_features_in_ = X.shape[1]

        self.spline_ = SplineTransformer(
            n_knots=self.n_knots,
            degree=self.degree,
            knots=self.knots,
            include_bias=True,
            extrapolation="continue",
        ).fit(X)

        basis = self._basis(X, (0,) * self.n_features_in_)

        if self.r_alpha is None:
            self.model_ = LinearRegression(
                fit_intercept=False
            ).fit(basis, y)
        else:
            self.model_ = Ridge(
                fit_intercept=False,
                alpha=self.r_alpha,
            ).fit(basis, y)

        return self

    def predict(self, X, alpha=None):
        check_is_fitted(self, "model_")
        X = check_array(X)

        alpha = (
            (0,) * self.n_features_in_
            if alpha is None
            else alpha
        )

        if len(alpha) != self.n_features_in_:
            raise ValueError("alpha must have one entry per feature")

        return self._basis(X, alpha) @ self.model_.coef_

In [4]:
dgp = "exponential"  # "sine", "quartic", or "exponential"

def f(x):
    u, v = x.T

    if dgp == "sine":
        su, sv = np.sin(2 * np.pi * u), np.sin(2 * np.pi * v)
        return su + sv + 0.5 * su * sv

    if dgp == "quartic":
        return (u**2 + v - 1)**2 + (u + v**2 - 1)**2

    if dgp == "exponential":
        return 2 * np.exp(2 * u * v - 2)

    raise ValueError(dgp)


def hessian_f(x):
    u, v = x.T
    H = np.empty((len(x), 2, 2))

    if dgp == "sine":
        w = 2 * np.pi
        su, sv = np.sin(w * u), np.sin(w * v)
        H[:, 0, 0] = -w**2 * su * (1 + 0.5 * sv)
        H[:, 1, 1] = -w**2 * sv * (1 + 0.5 * su)
        H[:, 0, 1] = H[:, 1, 0] = (
            0.5 * w**2 * np.cos(w * u) * np.cos(w * v)
        )

    elif dgp == "quartic":
        H[:, 0, 0] = 12 * u**2 + 4 * v - 2
        H[:, 1, 1] = 12 * v**2 + 4 * u - 2
        H[:, 0, 1] = H[:, 1, 0] = 4 * (u + v)

    elif dgp == "exponential":
        g = f(x)
        H[:, 0, 0] = 4 * v**2 * g
        H[:, 1, 1] = 4 * u**2 * g
        H[:, 0, 1] = H[:, 1, 0] = (2 + 4 * u * v) * g

    else:
        raise ValueError(dgp)

    return H


x_test_fixed = np.random.default_rng(2026).uniform(size=(50_000, 2))

def make_mc(seeds=range(100), n_train=1_000, sigma=0.2):
    mc = {}
    y_test = f(x_test_fixed)

    for seed in seeds:
        rng = np.random.default_rng(seed)
        x_train = rng.uniform(size=(n_train, 2))
        y_train = f(x_train) + rng.normal(scale=sigma, size=n_train)
        mc[seed] = (x_train, y_train, x_test_fixed, y_test)

    return mc

In [5]:
mse_20, mse_11, mse_02 = [[], []], [[], []], [[], []]
mae_20, mae_11, mae_02 = [[], []], [[], []], [[], []]

i = 0
for seed, data in make_mc(seeds=range(20), n_train=1000).items():
    print(i + 1)
    x_train, y_train, x_test, y_test = data
    tree = LinearTreeRegressor(
        base_estimator=Ridge(alpha=1e-2),
        #base_estimator=LinearRegression(),
        local_degree=3,
        derivative_degree=2,
        max_depth=3,
        min_samples_leaf=30,
        max_bins=25,
        max_features=2,
        random_state=seed,
        derivative_weights=[0.0,0.0],
    )

    forest = SobolevForestRegressor(
        estimator=tree,
        n_estimators=750,
        max_samples=0.7,
        n_jobs=-1,
        random_state=seed,
    )

    d = x_train.shape[1]
    n_knots = 3

    knots = np.tile(
        np.linspace(0, 1, n_knots)[:, None],
        (1, d),
    )

    spline = TensorSplineRegressor(
        degree=3,
        knots=knots,
        r_alpha=1e-3,
    ).fit(x_train, y_train)

    forest.fit(x_train, y_train)

    f_hat_20 = forest.predict(x_test, alpha=(2, 0))
    f_hat_11 = forest.predict(x_test, alpha=(1, 1))
    f_hat_02 = forest.predict(x_test, alpha=(0, 2))

    spline_hat_20 = spline.predict(x_test, alpha=(2, 0))
    spline_hat_11 = spline.predict(x_test, alpha=(1, 1))
    spline_hat_02 = spline.predict(x_test, alpha=(0, 2))

    hessian_test = hessian_f(x_test)

    d20_test = hessian_test[:, 0, 0]
    d11_test = hessian_test[:, 0, 1]
    d02_test = hessian_test[:, 1, 1]

    mse_20[0].append(((f_hat_20 - d20_test) ** 2).mean())
    mse_11[0].append(((f_hat_11 - d11_test) ** 2).mean())
    mse_02[0].append(((f_hat_02 - d02_test) ** 2).mean())
    mae_20[0].append(np.abs(f_hat_20 - d20_test).mean())
    mae_11[0].append(np.abs(f_hat_11 - d11_test).mean())
    mae_02[0].append(np.abs(f_hat_02 - d02_test).mean())

    mse_20[1].append(((spline_hat_20 - d20_test) ** 2).mean())
    mse_11[1].append(((spline_hat_11 - d11_test) ** 2).mean())
    mse_02[1].append(((spline_hat_02 - d02_test) ** 2).mean())
    mae_20[1].append(np.abs(spline_hat_20 - d20_test).mean())
    mae_11[1].append(np.abs(spline_hat_11 - d11_test).mean())
    mae_02[1].append(np.abs(spline_hat_02 - d02_test).mean())

    print('MSE forest:', np.mean(mse_20[0]), 'MAE forest:', np.mean(mae_20[0]))
    print('MSE spline:', np.mean(mse_20[1]), 'MAE spline:', np.mean(mae_20[1]))

    i += 1

1
MSE forest: 1.0092904733613257 MAE forest: 0.5913223068733827
MSE spline: 1.552190317588318 MAE spline: 0.7262086263712323
2
MSE forest: 0.7856081112458821 MAE forest: 0.5269872302017198
MSE spline: 0.9642572623549065 MAE spline: 0.6140638267391967
3
MSE forest: 0.8859245307746318 MAE forest: 0.5333923011945592
MSE spline: 1.02756526494114 MAE spline: 0.6532246456138345
4
MSE forest: 0.8168729317246722 MAE forest: 0.5298341318799191
MSE spline: 0.9783844022034955 MAE spline: 0.665461086002228
5
MSE forest: 0.7486944977430208 MAE forest: 0.49015910904183385
MSE spline: 0.9261697819491215 MAE spline: 0.6500866962536873
6


KeyboardInterrupt: 

In [ ]:
np.mean(mse_11[1])